# Part 2 - Naive Bayes (SMS Spam Detection)
**Gray Interface '26 | Task 3**

Dataset: [SMS Spam Collection](https://www.kaggle.com/datasets/abhishek14398/sms-spam-collection)

## Importing the dependencies

In [ ]:
!pip install kagglehub --quiet

In [ ]:
import kagglehub
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import string
import re

from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report)
from nltk.corpus import stopwords
import nltk
nltk.download('stopwords', quiet=True)


## Loading the Dataset

In [ ]:
path = kagglehub.dataset_download("abhishek14398/sms-spam-collection")
print(os.listdir(path))


In [ ]:
# The file is tab-separated with no header
df = pd.read_csv(os.path.join(path, "SMSSpamCollection"),
                 sep='\t', header=None, names=['label', 'message'],
                 encoding='latin-1')
print("Shape:", df.shape)
df.head()


## Exploratory Data Analysis

In [ ]:
# Class distribution
print(df['label'].value_counts())

plt.figure(figsize=(5, 4))
df['label'].value_counts().plot(kind='bar', color=['steelblue', '#E50914'])
plt.title('Class Distribution: Ham vs Spam')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
# Message length analysis
df['msg_length'] = df['message'].apply(len)

plt.figure(figsize=(9, 4))
df.groupby('label')['msg_length'].plot(kind='hist', bins=50, alpha=0.6, legend=True)
plt.title('Message Length Distribution: Ham vs Spam')
plt.xlabel('Message Length')
plt.tight_layout()
plt.show()

print(df.groupby('label')['msg_length'].mean())


## Text Preprocessing

In [ ]:
stop_words = set(stopwords.words('english'))

def clean_text(text):
    # Convert to lowercase
    text = text.lower()
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    # Remove numbers
    text = re.sub(r'\d+', '', text)
    # Remove stopwords
    tokens = text.split()
    tokens = [w for w in tokens if w not in stop_words]
    return ' '.join(tokens)

df['cleaned'] = df['message'].apply(clean_text)

# Preview
print("Original:", df['message'][0])
print("Cleaned :", df['cleaned'][0])


## Train-Test Split

In [ ]:
# Encode labels: ham=0, spam=1
df['label_enc'] = (df['label'] == 'spam').astype(int)

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    df['cleaned'], df['label_enc'], test_size=0.2, random_state=42, stratify=df['label_enc']
)
print(f"Train: {len(X_train_raw)}, Test: {len(X_test_raw)}")


## Experimenting with Vectorizers, N-grams and Alpha

In [ ]:
def run_experiment(vectorizer, X_tr, X_te, y_tr, y_te, alpha, label):
    X_tr_vec = vectorizer.fit_transform(X_tr)
    X_te_vec = vectorizer.transform(X_te)

    model = MultinomialNB(alpha=alpha)
    model.fit(X_tr_vec, y_tr)
    preds      = model.predict(X_te_vec)
    preds_prob = model.predict_proba(X_te_vec)[:, 1]

    return {
        'Config': label,
        'Alpha':  alpha,
        'Accuracy':  round(accuracy_score(y_te, preds), 4),
        'Precision': round(precision_score(y_te, preds), 4),
        'Recall':    round(recall_score(y_te, preds), 4),
        'F1':        round(f1_score(y_te, preds), 4),
        'ROC-AUC':   round(roc_auc_score(y_te, preds_prob), 4),
    }

results = []
alphas = [0.1, 0.5, 1.0, 2.0, 5.0]

for alpha in alphas:
    # CountVectorizer - Unigrams
    results.append(run_experiment(
        CountVectorizer(ngram_range=(1,1)),
        X_train_raw, X_test_raw, y_train, y_test,
        alpha, f'Count-Unigram'
    ))
    # CountVectorizer - Bigrams
    results.append(run_experiment(
        CountVectorizer(ngram_range=(1,2)),
        X_train_raw, X_test_raw, y_train, y_test,
        alpha, f'Count-Bigram'
    ))
    # TF-IDF - Unigrams
    results.append(run_experiment(
        TfidfVectorizer(ngram_range=(1,1)),
        X_train_raw, X_test_raw, y_train, y_test,
        alpha, f'TFIDF-Unigram'
    ))
    # TF-IDF - Bigrams
    results.append(run_experiment(
        TfidfVectorizer(ngram_range=(1,2)),
        X_train_raw, X_test_raw, y_train, y_test,
        alpha, f'TFIDF-Bigram'
    ))

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))


## Comparing Configs — F1 Score

In [ ]:
# Plot F1 for each config across alpha values
pivot = results_df.pivot_table(index='Alpha', columns='Config', values='F1')

pivot.plot(figsize=(10, 5), marker='o')
plt.title('F1 Score vs Alpha for Different Vectorizer Configurations')
plt.xlabel('Alpha (smoothing)')
plt.ylabel('F1 Score')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()


## Best Model — Confusion Matrix

In [ ]:
# Find best configuration by F1
best_row = results_df.loc[results_df['F1'].idxmax()]
print("Best config:", best_row['Config'], "| Alpha:", best_row['Alpha'])
print(best_row)

# Re-train best model
best_alpha  = best_row['Alpha']
best_config = best_row['Config']

if 'TFIDF' in best_config:
    ngram = (1,2) if 'Bigram' in best_config else (1,1)
    vec = TfidfVectorizer(ngram_range=ngram)
else:
    ngram = (1,2) if 'Bigram' in best_config else (1,1)
    vec = CountVectorizer(ngram_range=ngram)

X_tr_vec = vec.fit_transform(X_train_raw)
X_te_vec = vec.transform(X_test_raw)
best_model = MultinomialNB(alpha=best_alpha)
best_model.fit(X_tr_vec, y_train)
preds = best_model.predict(X_te_vec)

cm = confusion_matrix(y_test, preds)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred: Ham', 'Pred: Spam'],
            yticklabels=['True: Ham', 'True: Spam'])
plt.title(f'Confusion Matrix — {best_config} (alpha={best_alpha})')
plt.tight_layout()
plt.show()

print(classification_report(y_test, preds, target_names=['Ham', 'Spam']))


## Observations

- **CountVectorizer** treats all words equally by raw count. **TF-IDF** down-weights words that appear in many messages (like "the", "is") — giving more weight to distinctive spam words like "free", "win", "prize".
- **TF-IDF generally outperforms CountVectorizer** on this dataset because spam has distinctive vocabulary that TF-IDF rewards.
- **Bigrams** (pairs of words like "free entry", "prize winner") capture spam patterns better than unigrams alone — spam messages use specific phrase patterns.
- **Alpha (smoothing)**: Very low alpha (0.1) can overfit to training vocabulary. Alpha=1.0 is a good default. Very high alpha (5.0) smooths too aggressively and loses discriminative power.
- Spam messages are significantly longer on average — spammers pack more content to be convincing.
- The dataset is imbalanced (~87% ham, 13% spam) — Precision and Recall matter more than raw Accuracy.
